# Laboratorio 2 - Complejidad y búsqueda de hiperparámetros

En este laboratorio del caso AlpesHearth, se busca fortalecer el modelo inicial de regresión lineal desarrollado previamente para estimar el riesgo cardiovascular, explorando enfoques más avanzados de modelado y evaluando su capacidad de generalización y estabilidad. A partir del conjunto de datos ya depurado, el énfasis estará en la construcción y comparación de modelos de regresión polinomial y regularizada (Ridge y Lasso), utilizando pipelines y validación cruzada con búsqueda sistemática de hiperparámetros. Se analizará el impacto de la complejidad en el desempeño predictivo mediante métricas como RMSE, MAE y R², así como a través de curvas de validación y el estudio del trade-off sesgo-varianza. Posteriormente, se seleccionará el mejor modelo considerando no solo su desempeño promedio sino también su estabilidad, y se estimarán intervalos de confianza mediante bootstrapping para cuantificar la incertidumbre. Finalmente, se realizará un análisis cuantitativo, cualitativo y conceptual de los resultados, comunicando de manera clara las decisiones metodológicas y sus implicaciones técnicas y estratégicas para la organización.

## 1. Importar Librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
from sklearn.model_selection import validation_curve

## 2. Carga y preparación de los datos

### Carga de los datos:

In [2]:
datos_pacientes = pd.read_csv('./data/Datos Lab 1.csv')
datos = datos_pacientes.copy()
datos.head()

,Patient ID,Date of Service,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),...,Physical Activity Level,Family History of CVD,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score,CVD Risk Level
0,isDx5313,"November 08, 2023",M,44.0,114.300,1.720,38.600,100.000,112/83,228.0,...,High,N,172.000,0.581,112.0,83.0,Hypertension Stage 1,121.0,19.880,HIGH
1,LHCK2961,20/03/2024,F,57.0,92.923,1.842,33.116,106.315,101/91,158.0,...,High,Y,184.172,0.577,101.0,91.0,Hypertension Stage 2,57.0,16.833,INTERMEDIARY
2,WjVn1699,2021-05-27,F,NaN,73.400,1.650,27.000,78.100,90/74,135.0,...,High,N,165.000,0.473,90.0,74.0,Normal,45.0,12.600,LOW
3,dCDO1109,"April 18, 2022",F,35.0,113.300,1.780,35.800,79.600,92/89,158.0,...,Moderate,Y,178.000,0.447,92.0,89.0,Hypertension Stage 1,94.0,14.920,HIGH
4,pnpE1080,01/11/2024,F,48.0,102.200,1.750,33.400,106.700,121/68,207.0,...,Low,Y,175.000,0.610,121.0,68.0,Elevated,128.0,18.870,HIGH


### Preparación:

Se hace un tratamiento previo de unicidad antes del pipeline.

Eliminación de filas duplicadas.

In [3]:
datos.duplicated().sum()

np.int64(151)

In [4]:
datos = datos.drop_duplicates()

Eliminación de datos diferentes de mismos pacientes el mismo día.

In [5]:
datos["Date of Service"] = pd.to_datetime(
    datos["Date of Service"],
    format="mixed",   # permite múltiples formatos
    dayfirst=True,    # interpreta correctamente DD/MM/YYYY
    errors="coerce"   # valores inválidos → NaT
)

In [6]:
n_dups = datos.duplicated(subset=["Patient ID", "Date of Service"], keep="first").sum()
print("Filas duplicadas por (Patient ID, Date of Service):", int(n_dups))

Filas duplicadas por (Patient ID, Date of Service): 112


In [7]:
datos = (
    datos.sort_values(["Patient ID", "Date of Service"])
      .drop_duplicates(subset=["Patient ID", "Date of Service"], keep="first")
      .reset_index(drop=True)
)

In [8]:
datos.duplicated(
    subset=["Patient ID", "Date of Service"]
).sum()

np.int64(0)

## 3. Definición del problema modelado

### 3.1 Separación de variables

In [9]:
target = "CVD Risk Score"
X = datos.drop(columns=[target])
y = datos[target]

### 3.2 Partición de los datos

In [10]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.25, random_state=42)

In [11]:
X_train.shape, y_train.shape

((1032, 23), (1032,))

In [12]:
X_test.shape, y_test.shape

((344, 23), (344,))

## 4. Construcción del Pipeline de Preprocesamiento

### 4.1 Selección y exclusión de variables

In [13]:
cols_to_drop = [
    "Patient ID", "Date of Service", "Blood Pressure Category",
    "CVD Risk Level", "BMI", "Height (cm)",
    "Waist-to-Height Ratio", "CVD Risk Score"
]

def drop_columns(df):
    """Elimina columnas irrelevantes o redundantes del DataFrame."""
    return df.drop(columns=cols_to_drop, errors="ignore")

dropper = FunctionTransformer(drop_columns)

### 4.2 Separación por tipo de variable (numéricas y categóricas)

In [14]:
numeric_features = [
    "Age","Weight (kg)","Height (m)","BMI",
    "Abdominal Circumference (cm)","Total Cholesterol (mg/dL)",
    "HDL (mg/dL)","Fasting Blood Sugar (mg/dL)",
    "Systolic BP","Diastolic BP","Estimated LDL (mg/dL)"
]

categorical_features = [
    "Sex","Smoking Status","Diabetes Status",
    "Physical Activity Level","Family History of CVD"
]

### 4.3 Transformaciones de calidad de los datos

Cálculo de valores que se pueden conseguir  partir de las otras columnas.

In [ ]:
def calcular_derived(df):
    "BMI"
    df = df.copy()
    mask_bmi = (
        df['BMI'].isna()
        & df['Weight (kg)'].notna()
        & df['Height (m)'].notna()
    )
    df.loc[mask_bmi, 'BMI'] = (
        df.loc[mask_bmi, 'Weight (kg)']
        / (df.loc[mask_bmi, 'Height (m)'] ** 2)
    )

    "Waist to height ratio"
    mask_whtr = (
        df['Waist-to-Height Ratio'].isna()
        & df['Abdominal Circumference (cm)'].notna()
        & df['Height (cm)'].notna()
    )
    df.loc[mask_whtr, 'Waist-to-Height Ratio'] = (
        df.loc[mask_whtr, 'Abdominal Circumference (cm)']
        / df.loc[mask_whtr, 'Height (cm)']
    )
    return df

derived_calc = FunctionTransformer(calcular_derived)

Consistencia entre Sytolic y Diastolic BP.

In [ ]:
def bp_consistency(df):
    df = df.copy()
    if 'Systolic BP' in df.columns and 'Diastolic BP' in df.columns:
        mask = df['Systolic BP'] < df['Diastolic BP']
        df.loc[mask, ['Systolic BP', 'Diastolic BP']] = (
            df.loc[mask, ['Diastolic BP', 'Systolic BP']].values
        )
    return df

bp_check = FunctionTransformer(bp_consistency)

Usa rango intercuartil para decidir qué datos son atípicos.

In [ ]:
def iqr_filter(df, k=1.5):
    df = df.copy()
    num = df.select_dtypes(include=[np.number])
    Q1 = num.quantile(0.25)
    Q3 = num.quantile(0.75)
    IQR = Q3 - Q1
    mask = ~((num < (Q1 - k * IQR)) | (num > (Q3 + k * IQR))).any(axis=1)
    return df.loc[mask]

iqr_valid = FunctionTransformer(iqr_filter)

Pipeline de limpieza:

In [ ]:
pipeline_clean = Pipeline(steps=[
    ("derived", derived_calc),
    ("bp_check", bp_check),
    ("iqr_valid", iqr_valid),
    ("dropper", dropper),
])

After cleaning, shape: (968, 16)


Transformaciones finales para las variables numéricas y categóricas:

In [ ]:
dnumeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

catrans = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", dnumeric, numeric_features),
        ("cat", catrans, categorical_features),
    ]
)




Pipeline completo de limpieza y transformaciones:

In [ ]:
full_pipeline = Pipeline([
    ("clean", pipeline_clean),
    ("preproc", preprocessor),
])

### 4.4 Generación de características polinomiales

### 4.5 Ensamblaje del pipeline completo

## 5. Construcción del modelo de regresión polinomial

### 5.1 Definición del modelo base

### 5.2 Definición del espacio de hiperparámetros

### 5.3 Búsqueda de hiperparámetros con GridSearchCV

## 6. Evaluación del modelo polinomial

### 6.1 Mejor configuración encontrada

### 6.2 Predicciones sobre el conjunto de prueba

### 6.3 Reporte de métricas

## 7. Curvas de validación y análisis de complejidad

### 7.1 Generación de validation curves

### 7.2 Visualización del desempeño

### 7.3 Análisis sesgo-varianza

## 8. Modelo Ridge (Regularización L2)

### 8.1 Construcción del pipeline Ridge

### 8.2 Definición de hiperparámetros (α)

### 8.3 Búsqueda con GridSearchCV

### 8.4 Evaluación del modeo Ridge (Métricas)

## 9. Modelo Lasso (Regularización L1)

### 9.1 Construcción del pipeline Ridge

### 9.2 Definición de hiperparámetros (α)

### 9.3 Búsqueda con GridSearchCV

### 9.4 Evaluación del modeo Ridge (Métricas)

### 9.5 Análisis de Coeficientes

## 10. Construcción del modelo polinomial regularizado

### 10.1 Pipeline combinado

### 10.2 Espacio de hiperparámetros

### 10.3 Búsqueda con GridSearchCV

## 11. Evaluación del modelo polinomial regularizado

### 11.1 Mejor configuración encontrada

### 11.2 Evaluación en test (Métricas)

## 12. Comparación global de modelos

### 12.1 Tabla comparativa de resultados

### 12.2 Análisis de estabilidad vs desempeño

### 12.3 Selección del mejor modelo

### 12.4 Justificación metodológica de la elección

## 13. Construcción de intervalos de confianza (Bootstrapping)

### 13.1 Definición del procedimiento de remuestreo

### 13.2 Ejecución de boostrapping

### 13.3 Cálculo de intervalos de confianza

### 13.4 Visualización de distribuciones de error

### 13.5 Análisis de estabilidad y confiabilidad

## 14. Análisis cuantitativo

¿Cuál modelo obtuvo el mejor desempeño en el conjunto de test?

¿Coincide el mejor desempeño en test con el mejor promedio en validación cruzada? Si no coincide, ¿cuál puede ser la explicación?

¿El modelo con mejor métrica promedio es necesariamente el más adecuado? Justifica considerando también la desviación estándar del desempeño.

Con base en las curvas de validación, ¿cómo cambia el error a medida que aumenta la complejidad? ¿En qué punto se evidencia sobreajuste?

¿Cómo afecta la regularización la magnitud y estabilidad de los coeficientes?

¿Los intervalos de confianza obtenidos mediante bootstrapping sugieren estabilidad o alta variabilidad en el desempeño? ¿Qué implicaciones tiene esto?

## 15. Análisis cualitativo

¿Qué relación observas entre complejidad del modelo, capacidad de generalización y estabilidad del desempeño?

¿Qué fuentes de sesgo podrían estar presentes en los datos o en el proceso de modelado?

Si el tamaño de muestra fuera mayor, ¿esperarías cambios en la estabilidad de los modelos? Explique.